In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../churn.db")

### VISÃO GERAL DO PROBLEMA

##### Taxa de churn geral

In [3]:
query = """
SELECT COUNT(*) AS total_clientes,
    SUM(Churn) AS clientes_churn,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate_percent
FROM customers;
"""

df = pd.read_sql(query, conn)
df

,total_clientes,clientes_churn,churn_rate_percent
0,7043,1869,26.54


### ANÁLISE DE PERFIL

##### Churn por tipo de contrato

In [4]:
query = """
SELECT 
    Contract,
    COUNT(*) AS total,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate
FROM customers
GROUP BY Contract
ORDER BY churn_rate DESC;
"""

df = pd.read_sql(query, conn)
df

,Contract,total,churn_rate
0,Month-to-month,3875,42.71
1,One year,1473,11.27
2,Two year,1695,2.83


##### Churn por tempo de cliente (tenure)

In [5]:
query = """
SELECT 
    CASE 
        WHEN tenure < 12 THEN '0-1 ano'
        WHEN tenure < 24 THEN '1-2 anos'
        ELSE '2+ anos'
    END AS faixa_tempo,
    COUNT(*) AS total,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate
FROM customers
GROUP BY faixa_tempo
ORDER BY churn_rate DESC;
"""

df = pd.read_sql(query, conn)
df

,faixa_tempo,total,churn_rate
0,0-1 ano,2069,48.28
1,1-2 anos,1047,29.51
2,2+ anos,3927,14.29


### SERVIÇOS E COMPORTAMENTO

##### Churn por suporte técnico

In [6]:
query = """
SELECT 
    TechSupport,
    COUNT(*) AS total,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate
FROM customers
GROUP BY TechSupport
ORDER BY churn_rate DESC;
"""

df = pd.read_sql(query, conn)
df

,TechSupport,total,churn_rate
0,No,3473,41.64
1,Yes,2044,15.17
2,No internet service,1526,7.40


##### Churn por tipo de internet

In [7]:
query = """
SELECT InternetService,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate
FROM customers
GROUP BY InternetService
ORDER BY churn_rate DESC;
"""

df = pd.read_sql(query, conn)
df

,InternetService,churn_rate
0,Fiber optic,41.89
1,DSL,18.96
2,No,7.40


##### Churn por método de pagamento

In [8]:
query = """
SELECT 
    PaymentMethod,
    ROUND(AVG(Churn) * 100, 2) AS churn_rate
FROM customers
GROUP BY PaymentMethod
ORDER BY churn_rate DESC;
"""

df = pd.read_sql(query, conn)
df

,PaymentMethod,churn_rate
0,Electronic check,45.29
1,Mailed check,19.11
2,Bank transfer (automatic),16.71
3,Credit card (automatic),15.24


##### INSIGHT: Perfil de alto risco

In [9]:
query = """
SELECT *
FROM customers
WHERE 
    Contract = 'Month-to-month'
    AND tenure < 12
    AND Churn = 1;
"""

df = pd.read_sql(query, conn)
df

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
1,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1
2,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1
3,4190-MFLUW,Female,0,Yes,Yes,10,Yes,No,DSL,No,...,Yes,Yes,No,No,Month-to-month,No,Credit card (automatic),55.20,528.35,1
4,8779-QRDMV,Male,1,No,No,1,No,No phone service,DSL,No,...,Yes,No,No,Yes,Month-to-month,Yes,Electronic check,39.65,39.65,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
986,5482-NUPNA,Female,0,No,No,4,Yes,No,DSL,Yes,...,No,Yes,No,No,Month-to-month,Yes,Mailed check,60.40,272.15,1
987,1122-JWTJW,Male,0,Yes,Yes,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Mailed check,70.65,70.65,1
988,8775-CEBBJ,Female,0,No,No,9,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,Yes,Bank transfer (automatic),44.20,403.35,1
989,6894-LFHLY,Male,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,75.75,75.75,1
